In [ ]:
"""
Huff-based preferred LPG distributor assignment per inhabited 1 km pixel (Nigeria)
using SHORTEST-PATH travel time on a friction graph.

What this script does
---------------------
This script computes, for each inhabited pixel, the preferred LPG distributor for:
1) walking population
2) motorized population

The winner is selected with a Huff-style gravity score:

    score_ij = A_j / (T_ij + eps)^beta

where:
- A_j = distributor attractiveness
- T_ij = shortest-path travel time (minutes) from pixel i to distributor j
- beta = distance-decay exponent (default 2.0)
- eps = small stabilization constant

Important implementation choice
------------------------------
Travel time is computed from a raster graph using shortest-path (Dijkstra) on a
4-neighbor network weighted by friction (minutes per meter -> minutes per cell edge).
No straight-line approximation is used.

Scalability strategy
--------------------
To keep runtime manageable on large grids:
- process only inhabited pixels (from Population.tif)
- build candidates directly within 60 km, expanding to 100 km only when needed
- keep at least MIN_CANDIDATES = 4 (or nearest-neighbor safety in sparse areas)
- run one shortest-path per mode in the normal case
- use one final fallback shortest-path only as safety net
- winner is always selected by Huff score (attractiveness + travel time)

Outputs
-------
Main output: GPKG point layer with one point per inhabited pixel center and fields:
- car_share, walk_share
- best_reseller_id_walk, best_time_walk_min, best_place_id_walk
- best_reseller_id_car,  best_time_car_min,  best_place_id_car

Optional outputs: support rasters for winner ids and winner times.
"""

from __future__ import annotations

import time
from typing import Iterable
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

import rasterio
from rasterio.warp import reproject, Resampling

# =====================================================================
# USER PARAMETERS
# =====================================================================
DATA_DIR = "dataset_big"

# Inputs
RESELLER_GPKG = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"
CAR_SHARE_RASTER = f"{DATA_DIR}/vehicles_allocation_share.tif"
WALK_FRICTION_RASTER = f"{DATA_DIR}/friction_walk.tif"
MOTO_FRICTION_RASTER = f"{DATA_DIR}/friction_moto.tif"

# Outputs
OUTPUT_GPKG = f"{DATA_DIR}/huff_preferred_distributor_per_pixel.gpkg"
OUTPUT_GPKG_LAYER = "pixel_preference"

OUTPUT_RASTER_BEST_ID_WALK = f"{DATA_DIR}/huff_best_reseller_id_walk.tif"
OUTPUT_RASTER_BEST_TIME_WALK = f"{DATA_DIR}/huff_best_time_walk_min.tif"
OUTPUT_RASTER_BEST_ID_CAR = f"{DATA_DIR}/huff_best_reseller_id_car.tif"
OUTPUT_RASTER_BEST_TIME_CAR = f"{DATA_DIR}/huff_best_time_car_min.tif"
OUTPUT_LOOKUP_CSV = f"{DATA_DIR}/huff_reseller_lookup.csv"

# Preferred columns
FID_COLUMN = "fid"
PLACE_ID_COLUMN = "place_id"
ATTRACTIVENESS_COLUMN = "attractiveness"

# Fallback id construction
GEOM_ID_DECIMALS = 6

# Huff parameters
BETA = 2.0
EPS = 1e-6
MIN_ATTRACTIVENESS = 1e-6

# Population rules
MIN_POP_PER_PIXEL = 0.0

# Car-share interpretation
CAR_SHARE_IS_PERCENT = True
CAR_SHARE_ZERO_THRESHOLD = 1e-9

# Candidate strategy
MIN_CANDIDATES = 4
PRIMARY_SEARCH_RADIUS_KM = 60
MAX_SEARCH_RADIUS_KM = 100
EXTRA_CANDIDATES_CAR = 6

# Adaptive limits and safety fallback
INITIAL_LIMIT_FACTOR_WALK = 8.0
INITIAL_LIMIT_FACTOR_CAR = 10.0
FINAL_LIMIT_FACTOR_WALK = 14.0
FINAL_LIMIT_FACTOR_CAR = 16.0
LIMIT_MARGIN_MIN = 30.0
UNASSIGNED_TIME_MIN = 7000.0
UNASSIGNED_PLACE_ID = "UNASSIGNED"

# Graph assumptions
CELL_SIZE_METERS = 1000.0  # 1km grid
USE_8_NEIGHBORS = False

# Progress
PROGRESS_EVERY = 2000
MAX_PIXELS_DEBUG = None  # set int for quick testing

# Outputs flags
SAVE_SUPPORT_RASTERS = True

# Nodata conventions
NODATA_FLOAT = -9999.0
NODATA_INT = -1

# =====================================================================
# HELPERS
# =====================================================================
def _read_raster(path: str):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile


def _align_to_reference(path: str, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    with rasterio.open(path) as src:
        dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _safe_attractiveness(values: Iterable) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").astype(np.float64).to_numpy()
    arr = np.where(np.isfinite(arr), arr, MIN_ATTRACTIVENESS)
    arr = np.maximum(arr, MIN_ATTRACTIVENESS)
    return arr


def _to_fraction(arr: np.ndarray) -> np.ndarray:
    out = arr.astype(np.float32, copy=True)
    if CAR_SHARE_IS_PERCENT:
        out = out / 100.0
    out = np.where(np.isfinite(out), out, 0.0)
    out = np.clip(out, 0.0, 1.0)
    return out


def _eta(seconds: float) -> str:
    if (not np.isfinite(seconds)) or seconds < 0:
        return "n/a"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"


def _score_huff(a: float, tmin: float) -> float:
    if (not np.isfinite(tmin)) or (tmin <= 0):
        return -np.inf
    return float(a / ((tmin + EPS) ** BETA))


def _write_float_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype(np.float32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


def _write_int_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="int32", count=1, nodata=NODATA_INT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_INT).astype(np.int32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


def _build_stable_reseller_ids(gdf: gpd.GeoDataFrame) -> tuple[np.ndarray, str]:
    if FID_COLUMN in gdf.columns:
        fid_num = pd.to_numeric(gdf[FID_COLUMN], errors="coerce")
        if fid_num.notna().all() and fid_num.is_unique:
            return fid_num.astype(np.int64).to_numpy(), "fid"

    if PLACE_ID_COLUMN in gdf.columns:
        place = gdf[PLACE_ID_COLUMN].astype(str).fillna("").replace("None", "")
        if (place.str.len() > 0).all() and place.is_unique:
            unique_sorted = np.sort(place.unique())
            mapping = {v: i + 1 for i, v in enumerate(unique_sorted)}
            return place.map(mapping).astype(np.int64).to_numpy(), "place_id_rank"

    x = np.round(gdf.geometry.x.to_numpy(), GEOM_ID_DECIMALS)
    y = np.round(gdf.geometry.y.to_numpy(), GEOM_ID_DECIMALS)
    key = pd.Series([f"{xx:.{GEOM_ID_DECIMALS}f}_{yy:.{GEOM_ID_DECIMALS}f}" for xx, yy in zip(x, y)])
    unique_sorted = np.sort(key.unique())
    mapping = {v: i + 1 for i, v in enumerate(unique_sorted)}
    return key.map(mapping).astype(np.int64).to_numpy(), "geometry_rank"


def _candidate_idx_adaptive(r: int, c: int, tree: cKDTree, coords: np.ndarray) -> np.ndarray:
    idx = np.array([], dtype=np.int32)

    found_primary = tree.query_ball_point([r, c], r=PRIMARY_SEARCH_RADIUS_KM, p=2)
    if len(found_primary) > 0:
        idx = np.asarray(found_primary, dtype=np.int32)

    if idx.size < MIN_CANDIDATES:
        found_max = tree.query_ball_point([r, c], r=MAX_SEARCH_RADIUS_KM, p=2)
        if len(found_max) > 0:
            idx = np.unique(np.concatenate([idx, np.asarray(found_max, dtype=np.int32)]))

    if idx.size < MIN_CANDIDATES:
        k = min(max(MIN_CANDIDATES, EXTRA_CANDIDATES_CAR), len(coords))
        _, nn = tree.query([r, c], k=k)
        idx = np.unique(np.concatenate([idx, np.atleast_1d(nn).astype(np.int32)]))

    return idx


def _winner_from_dist(dist_row: np.ndarray, cand_idx: np.ndarray, reseller_node: np.ndarray, reseller_id: np.ndarray, reseller_attr: np.ndarray):
    cand_nodes = reseller_node[cand_idx]
    t = dist_row[cand_nodes]
    a = reseller_attr[cand_idx]
    scores = np.array([_score_huff(aa, tt) for aa, tt in zip(a, t)], dtype=np.float64)
    if np.all(~np.isfinite(scores)):
        return NODATA_INT, np.nan
    j_local = int(np.nanargmax(scores))
    rid = int(reseller_id[cand_idx[j_local]])
    tmin = float(t[j_local]) if np.isfinite(t[j_local]) else np.nan
    return rid, tmin


def _run_mode_with_fallback(
    src_node: int,
    r: int,
    c: int,
    graph: csr_matrix,
    friction_min: float,
    base_idx: np.ndarray,
    r_rows: np.ndarray,
    r_cols: np.ndarray,
    reseller_node: np.ndarray,
    reseller_id: np.ndarray,
    reseller_attr: np.ndarray,
    src_component: int,
    reseller_component: np.ndarray,
    initial_limit_factor: float,
    final_limit_factor: float,
    ):
    cand = np.unique(base_idx)
    if cand.size > 0:
        cand = cand[reseller_component[cand] == src_component]

    if cand.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN

    lb = np.hypot(r_rows[cand] - r, r_cols[cand] - c) * CELL_SIZE_METERS * max(friction_min, EPS)
    limit = float(np.nanmax(lb) * initial_limit_factor + LIMIT_MARGIN_MIN) if lb.size > 0 else np.inf
    dist_row = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit,
    )[0]
    rid, tmin = _winner_from_dist(dist_row, cand, reseller_node, reseller_id, reseller_attr)
    if rid >= 0 and np.isfinite(tmin):
        return rid, float(tmin)

    cand_global = np.where(reseller_component == src_component)[0].astype(np.int32)
    if cand_global.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN

    lb2 = np.hypot(r_rows[cand_global] - r, r_cols[cand_global] - c) * CELL_SIZE_METERS * max(friction_min, EPS)
    limit2 = float(np.nanmax(lb2) * final_limit_factor + LIMIT_MARGIN_MIN) if lb2.size > 0 else np.inf
    dist_row2 = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit2,
    )[0]
    rid2, tmin2 = _winner_from_dist(dist_row2, cand_global, reseller_node, reseller_id, reseller_attr)
    if rid2 >= 0 and np.isfinite(tmin2):
        return rid2, float(tmin2)

    return NODATA_INT, UNASSIGNED_TIME_MIN


# =====================================================================
# LOAD / ALIGN INPUTS
# =====================================================================
t0 = time.time()
print("[1/8] Loading population reference raster...")
pop, ref_profile = _read_raster(POPULATION_RASTER)
transform = ref_profile["transform"]
crs = ref_profile["crs"]
height, width = pop.shape
print(f"Grid: {width} x {height}, CRS={crs}")

print("[2/8] Aligning car share and frictions to reference grid...")
car_share_raw = _align_to_reference(CAR_SHARE_RASTER, ref_profile, Resampling.nearest)
walk_friction = _align_to_reference(WALK_FRICTION_RASTER, ref_profile, Resampling.bilinear)
moto_friction = _align_to_reference(MOTO_FRICTION_RASTER, ref_profile, Resampling.bilinear)

car_share = _to_fraction(car_share_raw)
walk_share = (1.0 - car_share).astype(np.float32)

walk_friction = np.where(walk_friction > 0, walk_friction, np.nan).astype(np.float32)
moto_friction = np.where(moto_friction > 0, moto_friction, np.nan).astype(np.float32)

walk_friction_min = float(np.nanpercentile(walk_friction[np.isfinite(walk_friction)], 5))
moto_friction_min = float(np.nanpercentile(moto_friction[np.isfinite(moto_friction)], 5))

print(f"Walk friction range (min/m): {np.nanmin(walk_friction):.6f} .. {np.nanmax(walk_friction):.6f}")
print(f"Moto friction range (min/m): {np.nanmin(moto_friction):.6f} .. {np.nanmax(moto_friction):.6f}")

# =====================================================================
# LOAD RESELLERS
# =====================================================================
print("[3/8] Loading reseller points...")
resellers = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
if resellers.empty:
    raise RuntimeError("Reseller layer is empty.")
if resellers.crs != crs:
    resellers = resellers.to_crs(crs)

if ATTRACTIVENESS_COLUMN not in resellers.columns:
    raise KeyError(f"Missing column '{ATTRACTIVENESS_COLUMN}' in reseller layer.")

resellers = resellers[resellers.geometry.notna()].copy()
resellers = resellers[resellers.geometry.geom_type.isin(["Point"])].copy()
if resellers.empty:
    raise RuntimeError("No point geometries in reseller layer.")

resellers[ATTRACTIVENESS_COLUMN] = _safe_attractiveness(resellers[ATTRACTIVENESS_COLUMN])

r_rows, r_cols = rasterio.transform.rowcol(
    transform, resellers.geometry.x.values, resellers.geometry.y.values
 )
r_rows = np.asarray(r_rows, dtype=np.int32)
r_cols = np.asarray(r_cols, dtype=np.int32)

inside = (r_rows >= 0) & (r_rows < height) & (r_cols >= 0) & (r_cols < width)
resellers = resellers.loc[inside].copy()
r_rows = r_rows[inside]
r_cols = r_cols[inside]

if resellers.empty:
    raise RuntimeError("No reseller is inside raster extent.")

reseller_id, id_strategy = _build_stable_reseller_ids(resellers)
reseller_id = reseller_id.astype(np.int32)
print(f"Reseller id strategy: {id_strategy}")

reseller_attr = resellers[ATTRACTIVENESS_COLUMN].astype(np.float64).to_numpy()
if PLACE_ID_COLUMN in resellers.columns:
    reseller_place = resellers[PLACE_ID_COLUMN].astype(str).fillna("").to_numpy()
else:
    reseller_place = np.array([f"id_{v}" for v in reseller_id], dtype=object)

coords_rc = np.column_stack([r_rows, r_cols]).astype(np.float64)
reseller_tree = cKDTree(coords_rc)
print(f"Resellers on grid: {len(resellers)}")

# =====================================================================
# BUILD SHORTEST-PATH GRAPHS (walk and motorized)
# =====================================================================
print("[4/8] Building graph topology from valid cells...")
valid = np.isfinite(pop) & np.isfinite(walk_friction) & np.isfinite(moto_friction)
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
n_nodes = len(vr)
if n_nodes == 0:
    raise RuntimeError("No valid nodes available for graph.")
node_id[vr, vc] = np.arange(n_nodes, dtype=np.int32)
print(f"Valid graph nodes: {n_nodes:,}")

if USE_8_NEIGHBORS:
    neighbors = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
else:
    neighbors = [(-1,0),(1,0),(0,-1),(0,1)]

edge_i = []
edge_j = []
edge_cost_walk = []
edge_cost_moto = []

diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    fw0 = walk_friction[r, c]
    fm0 = moto_friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        fw1 = walk_friction[rr, cc]
        fm1 = moto_friction[rr, cc]
        if (not np.isfinite(fw1)) or (not np.isfinite(fm1)):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        cw = 0.5 * (fw0 + fw1) * step_m
        cm = 0.5 * (fm0 + fm1) * step_m
        if cw <= 0 or cm <= 0:
            continue

        edge_i.append(n0)
        edge_j.append(n1)
        edge_cost_walk.append(float(cw))
        edge_cost_moto.append(float(cm))

graph_walk = csr_matrix((edge_cost_walk, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
graph_moto = csr_matrix((edge_cost_moto, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
print(f"Directed edges: {len(edge_i):,}")

n_comp_walk, comp_walk = connected_components(csgraph=graph_walk, directed=False, return_labels=True)
n_comp_moto, comp_moto = connected_components(csgraph=graph_moto, directed=False, return_labels=True)
print(f"Connected components (walk/moto): {n_comp_walk:,} / {n_comp_moto:,}")

# map resellers to graph nodes
reseller_node = node_id[r_rows, r_cols]
valid_res = reseller_node >= 0
if not np.any(valid_res):
    raise RuntimeError("No reseller mapped to valid graph node.")
reseller_node = reseller_node[valid_res]
reseller_id = reseller_id[valid_res]
reseller_attr = reseller_attr[valid_res]
reseller_place = reseller_place[valid_res]
r_rows = r_rows[valid_res]
r_cols = r_cols[valid_res]
coords_rc = np.column_stack([r_rows, r_cols]).astype(np.float64)
reseller_tree = cKDTree(coords_rc)
reseller_comp_walk = comp_walk[reseller_node]
reseller_comp_moto = comp_moto[reseller_node]

# =====================================================================
# INHABITED PIXELS
# =====================================================================
print("[5/8] Preparing inhabited pixel list...")
inhabited = np.isfinite(pop) & (pop > MIN_POP_PER_PIXEL) & valid
pix_rows, pix_cols = np.where(inhabited)
n_pix = len(pix_rows)
if n_pix == 0:
    raise RuntimeError("No inhabited pixels found.")
if MAX_PIXELS_DEBUG is not None:
    n_use = min(MAX_PIXELS_DEBUG, n_pix)
    pix_rows = pix_rows[:n_use]
    pix_cols = pix_cols[:n_use]
    n_pix = n_use
    print(f"DEBUG mode active: {n_pix} pixels")
print(f"Inhabited pixels to process: {n_pix:,}")

best_id_walk = np.full((height, width), NODATA_INT, dtype=np.int32)
best_time_walk = np.full((height, width), np.nan, dtype=np.float32)
best_id_car = np.full((height, width), NODATA_INT, dtype=np.int32)
best_time_car = np.full((height, width), np.nan, dtype=np.float32)

id_to_place = {int(i): str(p) for i, p in zip(reseller_id, reseller_place)}

# =====================================================================
# MAIN LOOP (shortest-path per pixel source)
# =====================================================================
print("[6/8] Running Huff assignment with shortest-path travel times...")
loop_t0 = time.time()

for k, (r, c) in enumerate(zip(pix_rows, pix_cols), start=1):
    src_node = node_id[r, c]
    if src_node < 0:
        continue

    cand_walk = _candidate_idx_adaptive(int(r), int(c), reseller_tree, coords_rc)
    src_comp_walk = int(comp_walk[int(src_node)])

    rid_w, t_w = _run_mode_with_fallback(
        src_node=int(src_node),
        r=int(r),
        c=int(c),
        graph=graph_walk,
        friction_min=walk_friction_min,
        base_idx=cand_walk,
        r_rows=r_rows,
        r_cols=r_cols,
        reseller_node=reseller_node,
        reseller_id=reseller_id,
        reseller_attr=reseller_attr,
        src_component=src_comp_walk,
        reseller_component=reseller_comp_walk,
        initial_limit_factor=INITIAL_LIMIT_FACTOR_WALK,
        final_limit_factor=FINAL_LIMIT_FACTOR_WALK,
    )
    best_id_walk[r, c] = rid_w
    best_time_walk[r, c] = t_w

    cs = float(car_share[r, c])
    if cs <= CAR_SHARE_ZERO_THRESHOLD:
        best_id_car[r, c] = rid_w
        best_time_car[r, c] = t_w
    else:
        k_extra = min(EXTRA_CANDIDATES_CAR, len(coords_rc))
        _, nn = reseller_tree.query([int(r), int(c)], k=k_extra)
        nn = np.atleast_1d(nn).astype(np.int32)
        cand_car = np.unique(np.concatenate([cand_walk, nn]))
        src_comp_moto = int(comp_moto[int(src_node)])

        rid_c, t_c = _run_mode_with_fallback(
            src_node=int(src_node),
            r=int(r),
            c=int(c),
            graph=graph_moto,
            friction_min=moto_friction_min,
            base_idx=cand_car,
            r_rows=r_rows,
            r_cols=r_cols,
            reseller_node=reseller_node,
            reseller_id=reseller_id,
            reseller_attr=reseller_attr,
            src_component=src_comp_moto,
            reseller_component=reseller_comp_moto,
            initial_limit_factor=INITIAL_LIMIT_FACTOR_CAR,
            final_limit_factor=FINAL_LIMIT_FACTOR_CAR,
        )
        best_id_car[r, c] = rid_c
        best_time_car[r, c] = t_c

    if (k % PROGRESS_EVERY == 0) or (k == n_pix):
        elapsed = time.time() - loop_t0
        speed = k / max(elapsed, 1e-9)
        rem = (n_pix - k) / max(speed, 1e-9)
        total_est = elapsed + rem
        if total_est > 300:
            print(f"  {k:,}/{n_pix:,} ({100.0*k/n_pix:.1f}%) | {speed:.1f} pix/s | ETA {_eta(rem)}")
        else:
            print(f"  {k:,}/{n_pix:,} ({100.0*k/n_pix:.1f}%) | {speed:.1f} pix/s")

# =====================================================================
# OUTPUTS
# =====================================================================
print("[7/8] Writing GPKG output...")
out_rows = pix_rows.astype(np.int32)
out_cols = pix_cols.astype(np.int32)
xs = transform.c + (out_cols + 0.5) * transform.a
ys = transform.f + (out_rows + 0.5) * transform.e

df = pd.DataFrame({
    "row": out_rows,
    "col": out_cols,
    "x": xs.astype(np.float64),
    "y": ys.astype(np.float64),
    "car_share": car_share[out_rows, out_cols].astype(np.float32),
    "walk_share": walk_share[out_rows, out_cols].astype(np.float32),
    "best_reseller_id_walk": best_id_walk[out_rows, out_cols].astype(np.int32),
    "best_time_walk_min": best_time_walk[out_rows, out_cols].astype(np.float32),
    "best_reseller_id_car": best_id_car[out_rows, out_cols].astype(np.int32),
    "best_time_car_min": best_time_car[out_rows, out_cols].astype(np.float32),
})
df["best_place_id_walk"] = df["best_reseller_id_walk"].map(id_to_place).fillna(UNASSIGNED_PLACE_ID)
df["best_place_id_car"] = df["best_reseller_id_car"].map(id_to_place).fillna(UNASSIGNED_PLACE_ID)

gdf_out = gpd.GeoDataFrame(
    df,
    geometry=[Point(x, y) for x, y in zip(df["x"].to_numpy(), df["y"].to_numpy())],
    crs=crs,
 )
gdf_out.to_file(OUTPUT_GPKG, layer=OUTPUT_GPKG_LAYER, driver="GPKG")
print(f"Saved: {OUTPUT_GPKG} | layer={OUTPUT_GPKG_LAYER}")

lookup = pd.DataFrame({
    "reseller_id": reseller_id,
    "place_id": reseller_place,
    "attractiveness": reseller_attr,
    "row": r_rows,
    "col": r_cols,
    "id_strategy": id_strategy,
}).drop_duplicates(subset=["reseller_id"])
lookup.to_csv(OUTPUT_LOOKUP_CSV, index=False)
print(f"Saved lookup: {OUTPUT_LOOKUP_CSV}")

print("[8/8] Writing support rasters...")
if SAVE_SUPPORT_RASTERS:
    _write_int_raster(OUTPUT_RASTER_BEST_ID_WALK, best_id_walk.astype(np.float32), ref_profile)
    _write_float_raster(OUTPUT_RASTER_BEST_TIME_WALK, best_time_walk, ref_profile)
    _write_int_raster(OUTPUT_RASTER_BEST_ID_CAR, best_id_car.astype(np.float32), ref_profile)
    _write_float_raster(OUTPUT_RASTER_BEST_TIME_CAR, best_time_car, ref_profile)
    print("Support rasters saved.")
else:
    print("Support rasters skipped.")

valid_walk = (best_id_walk[inhabited] >= 0) & np.isfinite(best_time_walk[inhabited]) & (best_time_walk[inhabited] < UNASSIGNED_TIME_MIN)
valid_car = (best_id_car[inhabited] >= 0) & np.isfinite(best_time_car[inhabited]) & (best_time_car[inhabited] < UNASSIGNED_TIME_MIN)
print("\n=== SUMMARY ===")
print(f"Pixels processed: {n_pix:,}")
print(f"Walk assigned pixels: {int(valid_walk.sum()):,} ({100.0 * valid_walk.mean():.2f}%)")
print(f"Car assigned pixels : {int(valid_car.sum()):,} ({100.0 * valid_car.mean():.2f}%)")
if valid_walk.any():
    w = best_time_walk[inhabited][valid_walk]
    print(f"Walk time min/median/max (min): {np.nanmin(w):.2f} / {np.nanmedian(w):.2f} / {np.nanmax(w):.2f}")
if valid_car.any():
    c = best_time_car[inhabited][valid_car]
    print(f"Car  time min/median/max (min): {np.nanmin(c):.2f} / {np.nanmedian(c):.2f} / {np.nanmax(c):.2f}")

print(f"\nDone in {_eta(time.time() - t0)}")

[1/8] Loading population reference raster...
Grid: 1337 x 1085, CRS=EPSG:3857
[2/8] Aligning car share and frictions to reference grid...
Walk friction range (min/m): 0.012000 .. 0.162533
Moto friction range (min/m): 0.000500 .. 0.159416
[3/8] Loading reseller points...
Reseller id strategy: place_id_rank
Resellers on grid: 2792
[4/8] Building graph topology from valid cells...
Valid graph nodes: 563,482
Directed edges: 1,841,278
Connected components (walk/moto): 11,838 / 11,838
[5/8] Preparing inhabited pixel list...
Inhabited pixels to process: 563,482
[6/8] Running Huff assignment with shortest-path travel times...
  2,000/563,482 (0.4%) | 271.4 pix/s | ETA 34m 29s
  4,000/563,482 (0.7%) | 318.2 pix/s | ETA 29m 18s
  6,000/563,482 (1.1%) | 316.3 pix/s | ETA 29m 22s
  8,000/563,482 (1.4%) | 215.2 pix/s | ETA 43m 1s
  10,000/563,482 (1.8%) | 151.1 pix/s | ETA 1h 1m 1s
  12,000/563,482 (2.1%) | 118.8 pix/s | ETA 1h 17m 21s
  14,000/563,482 (2.5%) | 101.5 pix/s | ETA 1h 30m 13s
  16,000

In [2]:
# Diagnostic summary for Huff assignment outputs
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

DATA_DIR = "dataset_big"
OUT_GPKG = f"{DATA_DIR}/huff_preferred_distributor_per_pixel.gpkg"
OUT_LAYER = "pixel_preference"
LOOKUP_CSV = f"{DATA_DIR}/huff_reseller_lookup.csv"
POP_RASTER = f"{DATA_DIR}/Population.tif"
UNASSIGNED_TIME_MIN = 9999.0

df = gpd.read_file(OUT_GPKG, layer=OUT_LAYER)
if df.empty:
    raise ValueError("Output layer is empty.")

walk_id_col = "best_reseller_id_walk" if "best_reseller_id_walk" in df.columns else "best_fid_walk"
car_id_col = "best_reseller_id_car" if "best_reseller_id_car" in df.columns else "best_fid_car"

# Basic validity masks
valid_walk = np.isfinite(df["best_time_walk_min"]) & (df["best_time_walk_min"] > 0) & (df["best_time_walk_min"] < UNASSIGNED_TIME_MIN) & (df[walk_id_col] >= 0)
valid_car = np.isfinite(df["best_time_car_min"]) & (df["best_time_car_min"] > 0) & (df["best_time_car_min"] < UNASSIGNED_TIME_MIN) & (df[car_id_col] >= 0)
has_car = df["car_share"].fillna(0).to_numpy() > 0
no_car = ~has_car

print("=== BASIC COVERAGE ===")
print(f"Pixels in output table: {len(df):,}")
print(f"Walk assignment valid: {int(valid_walk.sum()):,} ({valid_walk.mean()*100:.2f}%)")
print(f"Car assignment valid : {int(valid_car.sum()):,} ({valid_car.mean()*100:.2f}%)")
print(f"Pixels with car_share > 0: {int(has_car.sum()):,} ({has_car.mean()*100:.2f}%)")
print(f"Pixels with car_share = 0: {int(no_car.sum()):,} ({no_car.mean()*100:.2f}%)")

print("\n=== UNASSIGNED BREAKDOWN ===")
unassigned_walk = ~valid_walk
unassigned_car = ~valid_car
print(f"Unassigned walk pixels: {int(unassigned_walk.sum()):,}")
print(f"Unassigned car pixels : {int(unassigned_car.sum()):,}")
print(f"Unassigned car among car_share > 0: {int((unassigned_car & has_car).sum()):,}")
print(f"Unassigned car among car_share = 0: {int((unassigned_car & no_car).sum()):,}")

def _pctl(series, name):
    vals = series[np.isfinite(series)].to_numpy()
    if vals.size == 0:
        print(f"{name}: no valid values")
        return
    p = np.percentile(vals, [5, 25, 50, 75, 95])
    print(
        f"{name} (min): p05={p[0]:.2f}, p25={p[1]:.2f}, p50={p[2]:.2f}, "
        f"p75={p[3]:.2f}, p95={p[4]:.2f}, max={vals.max():.2f}"
    )

print("\n=== TRAVEL TIME DISTRIBUTION (VALID ONLY) ===")
_pctl(df.loc[valid_walk, "best_time_walk_min"], "Walk time")
_pctl(df.loc[valid_car, "best_time_car_min"], "Car time")

# Compare walk vs car winners where both are valid and car share is positive
both_valid = valid_walk & valid_car & has_car
same_winner = (df.loc[both_valid, walk_id_col].to_numpy() == df.loc[both_valid, car_id_col].to_numpy()) if both_valid.any() else np.array([], dtype=bool)

print("\n=== WALK vs CAR WINNER COMPARISON (car_share > 0, both valid) ===")
print(f"Comparable pixels: {int(both_valid.sum()):,}")
if both_valid.any():
    print(f"Same winner walk/car: {int(same_winner.sum()):,} ({same_winner.mean()*100:.2f}%)")
    print(f"Different winner     : {int((~same_winner).sum()):,} ({(~same_winner).mean()*100:.2f}%)")
else:
    print("No comparable pixels.")

# ---------------------------------------------------------------------
# Radius diagnostics: is 60 km enough? How often 100 km or >100 km?
# Distances are Euclidean on grid (km), based on pixel row/col and lookup row/col.
# ---------------------------------------------------------------------
print("\n=== CANDIDATE RADIUS DIAGNOSTICS (using winner distance) ===")
if {"row", "col", walk_id_col, car_id_col}.issubset(df.columns):
    lookup = pd.read_csv(LOOKUP_CSV)
    if {"reseller_id", "row", "col"}.issubset(lookup.columns):
        lookup_rc = lookup[["reseller_id", "row", "col"]].drop_duplicates("reseller_id").copy()
        lookup_rc = lookup_rc.rename(columns={"row": "r_res", "col": "c_res"})

        base = df[["row", "col", walk_id_col, car_id_col]].copy()

        walk_join = base.merge(
            lookup_rc,
            left_on=walk_id_col,
            right_on="reseller_id",
            how="left",
        )
        d_walk_km = np.hypot(
            walk_join["row"].to_numpy(dtype=np.float64) - walk_join["r_res"].to_numpy(dtype=np.float64),
            walk_join["col"].to_numpy(dtype=np.float64) - walk_join["c_res"].to_numpy(dtype=np.float64),
        )

        car_join = base.merge(
            lookup_rc,
            left_on=car_id_col,
            right_on="reseller_id",
            how="left",
        )
        d_car_km = np.hypot(
            car_join["row"].to_numpy(dtype=np.float64) - car_join["r_res"].to_numpy(dtype=np.float64),
            car_join["col"].to_numpy(dtype=np.float64) - car_join["c_res"].to_numpy(dtype=np.float64),
        )

        def _print_radius_stats(label, dist_km, valid_mask):
            valid_dist = valid_mask & np.isfinite(dist_km)
            n = int(valid_dist.sum())
            print(f"{label} valid with distance: {n:,}")
            if n == 0:
                print("  no valid assigned pixels")
                return

            d = dist_km[valid_dist]
            n60 = int(np.sum(d <= 60.0))
            n100 = int(np.sum((d > 60.0) & (d <= 100.0)))
            n_over = int(np.sum(d > 100.0))
            print(f"  <=60 km   : {n60:,} ({100.0*n60/n:.2f}%)")
            print(f"  60-100 km : {n100:,} ({100.0*n100/n:.2f}%)")
            print(f"  >100 km   : {n_over:,} ({100.0*n_over/n:.2f}%)")
            print(f"  median km : {float(np.median(d)):.2f}")
            print(f"  p95 km    : {float(np.percentile(d, 95)):.2f}")

        _print_radius_stats("Walk", d_walk_km, valid_walk.to_numpy())
        _print_radius_stats("Car ", d_car_km, valid_car.to_numpy())

        print("\nInterpretation hint:")
        print("- high % <=60 km => 60 km is usually enough")
        print("- high % 60-100 km => expansion to 100 km is often needed")
        print("- high % >100 km => many winners come from beyond 100 km / safety fallback")
    else:
        print("Lookup CSV missing required columns: reseller_id, row, col")
else:
    print("Missing columns for radius diagnostics.")

# Population-weighted diagnostics
if {"row", "col", "car_share", "walk_share"}.issubset(df.columns):
    with rasterio.open(POP_RASTER) as src:
        pop = src.read(1).astype(np.float32)
        nod = src.nodata
    if nod is not None:
        pop = np.where(pop == nod, np.nan, pop)

    rr = df["row"].to_numpy(dtype=np.int32)
    cc = df["col"].to_numpy(dtype=np.int32)
    inside = (rr >= 0) & (rr < pop.shape[0]) & (cc >= 0) & (cc < pop.shape[1])

    pop_px = np.full(len(df), np.nan, dtype=np.float32)
    pop_px[inside] = pop[rr[inside], cc[inside]]
    pop_px = np.where(np.isfinite(pop_px), np.maximum(pop_px, 0.0), 0.0)

    people_walk = pop_px * df["walk_share"].fillna(0).to_numpy(dtype=np.float32)
    people_car = pop_px * df["car_share"].fillna(0).to_numpy(dtype=np.float32)

    print("\n=== POPULATION-WEIGHTED CHECK ===")
    print(f"Total population on output pixels: {float(np.nansum(pop_px)):,.0f}")
    print(f"Weighted walk population         : {float(np.nansum(people_walk)):,.0f}")
    print(f"Weighted car population          : {float(np.nansum(people_car)):,.0f}")

    # Top assigned distributors by weighted people
    tmp_walk = pd.DataFrame({
        "reseller_id": df[walk_id_col],
        "people": people_walk,
        "valid": valid_walk,
    })
    tmp_walk = tmp_walk[tmp_walk["valid"] & (tmp_walk["reseller_id"] >= 0)]
    top_walk = tmp_walk.groupby("reseller_id", as_index=False)["people"].sum().sort_values("people", ascending=False).head(10)

    tmp_car = pd.DataFrame({
        "reseller_id": df[car_id_col],
        "people": people_car,
        "valid": valid_car,
    })
    tmp_car = tmp_car[tmp_car["valid"] & (tmp_car["reseller_id"] >= 0)]
    top_car = tmp_car.groupby("reseller_id", as_index=False)["people"].sum().sort_values("people", ascending=False).head(10)

    print("\nTop 10 distributors by assigned WALK population")
    print(top_walk.to_string(index=False))

    print("\nTop 10 distributors by assigned CAR population")
    print(top_car.to_string(index=False))
else:
    print("\nPopulation-weighted diagnostics skipped (row/col/share columns missing).")

=== BASIC COVERAGE ===
Pixels in output table: 563,482
Walk assignment valid: 481,460 (85.44%)
Car assignment valid : 481,340 (85.42%)
Pixels with car_share > 0: 560,269 (99.43%)
Pixels with car_share = 0: 3,213 (0.57%)

=== UNASSIGNED BREAKDOWN ===
Unassigned walk pixels: 82,022
Unassigned car pixels : 82,142
Unassigned car among car_share > 0: 81,066
Unassigned car among car_share = 0: 1,076

=== TRAVEL TIME DISTRIBUTION (VALID ONLY) ===
Walk time (min): p05=51.44, p25=219.49, p50=454.99, p75=840.26, p95=1869.58, max=6085.82
Car time (min): p05=6.63, p25=98.08, p50=305.56, p75=678.91, p95=1589.30, max=6085.82

=== WALK vs CAR WINNER COMPARISON (car_share > 0, both valid) ===
Comparable pixels: 479,196
Same winner walk/car: 432,775 (90.31%)
Different winner     : 46,421 (9.69%)

=== CANDIDATE RADIUS DIAGNOSTICS (using winner distance) ===
Walk valid with distance: 481,460
  <=60 km   : 419,772 (87.19%)
  60-100 km : 47,423 (9.85%)
  >100 km   : 14,265 (2.96%)
  median km : 24.19
  p95